In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch

from datetime import datetime

In [4]:
# Canonical paths
base_dir = "/content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M"
preprocessed_dir = f"{base_dir}/AntM2C_preprocessed"
checkpoints_dir = f"{base_dir}/checkpoints"
splits_dir = f"{preprocessed_dir}/splits"
split_features_dir = f"{preprocessed_dir}/split_features"

results_dir = f"{base_dir}/eval_results"
explain_dir = os.path.join(results_dir, "explainability")
os.makedirs(explain_dir, exist_ok=True)

In [5]:
print("--- Paths resolved: ---")
print(" base_dir:", base_dir)
print(" preprocessed_dir:", preprocessed_dir)
print(" checkpoints_dir:", checkpoints_dir)
print(" results_dir:", results_dir)
print(" explain_dir:", explain_dir)

--- Paths resolved: ---
 base_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M
 preprocessed_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/AntM2C_preprocessed
 checkpoints_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/checkpoints
 results_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/eval_results
 explain_dir: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/eval_results/explainability


In [6]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Determinism flags (safe defaults)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [7]:
# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cpu


In [8]:
# Load configs + checkpoint + predictions
fusion_config_path = os.path.join(checkpoints_dir, "fusion_config.json")
meta_config_path = os.path.join(checkpoints_dir, "meta_config.json")
model_ckpt_path = os.path.join(checkpoints_dir, "coldfusionx_model_epoch3.pt")

pred_path = os.path.join(results_dir, "test_predictions.npz")

In [9]:
# Load JSON configs
with open(fusion_config_path, "r") as f:
    fusion_config = json.load(f)

with open(meta_config_path, "r") as f:
    meta_config = json.load(f)

# Load predictions (Notebook 09 output)
pred_data = np.load(pred_path)
pred_keys = list(pred_data.keys())

In [10]:
print("--- Loaded Artifacts ---")
print("fusion_config_path:", fusion_config_path)
print("meta_config_path:", meta_config_path)
print("model_ckpt_path:", model_ckpt_path)
print("pred_path:", pred_path)
print("pred_keys:", pred_keys)

# Print key values (audit-friendly)
print("\n--- Key Config Values ---")
print("d_model:", fusion_config.get("d_model"))
print("hidden_dim:", fusion_config.get("hidden_dim"))
print("num_experts:", fusion_config.get("num_experts"))
print("num_scenes:", fusion_config.get("num_scenes"))
print("FEWSHOT_N:", meta_config.get("FEWSHOT_N"))

--- Loaded Artifacts ---
fusion_config_path: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/checkpoints/fusion_config.json
meta_config_path: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/checkpoints/meta_config.json
model_ckpt_path: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/checkpoints/coldfusionx_model_epoch3.pt
pred_path: /content/drive/MyDrive/Final_Year_Project/Implementation/datasets/AntM2C_10M/eval_results/test_predictions.npz
pred_keys: ['y_true', 'y_logit', 'y_prob', 'cold_mask', 'scene_idx', 'alpha', 'gates']

--- Key Config Values ---
d_model: 256
hidden_dim: 256
num_experts: 4
num_scenes: 5
FEWSHOT_N: 10


In [11]:
# Quick sanity on predictions shapes
print("--- Prediction Array Shapes ---")
for k in ["y_true", "y_logit", "y_prob", "cold_mask", "scene_idx", "alpha"]:
    if k in pred_data:
        arr = pred_data[k]
        print(f"{k:10s}: {arr.shape} | dtype={arr.dtype}")

--- Prediction Array Shapes ---
y_true    : (373033,) | dtype=float32
y_logit   : (373033,) | dtype=float32
y_prob    : (373033,) | dtype=float32
cold_mask : (373033,) | dtype=bool
scene_idx : (373033,) | dtype=int64
alpha     : (373033, 1) | dtype=float32


 #### Load Predictions + Build a “Selection Table”

In [12]:
# Load arrays from test_predictions.npz
y_true = pred_data["y_true"].astype(np.float32)
y_logit = pred_data["y_logit"].astype(np.float32)
y_prob = pred_data["y_prob"].astype(np.float32)
cold_mask = pred_data["cold_mask"].astype(bool)
scene_idx = pred_data["scene_idx"].astype(np.int64)
alpha = pred_data["alpha"].astype(np.float32).squeeze()   # (N,)

In [13]:
# gates optional
gates = pred_data["gates"].astype(np.float32) if "gates" in pred_data else None  # (N, E)

In [14]:
N = len(y_true)
assert alpha.shape == (N,), f"alpha shape mismatch: {alpha.shape} vs {(N,)}"

In [15]:
# row_idx mapping
test_idx_path = os.path.join(splits_dir, "test_idx.npy")
test_idx = np.load(test_idx_path).astype(np.int64)  # row indices into row-aligned feature stores
assert test_idx.shape[0] == N, f"test_idx mismatch: {test_idx.shape[0]} vs N={N}"

In [16]:
# Basic DataFrame
pred_df = pd.DataFrame({
    "row_idx": test_idx,
    "y_true": y_true,
    "y_logit": y_logit,
    "y_prob": y_prob,
    "cold_mask": cold_mask,
    "scene_idx": scene_idx,
    "alpha": alpha,
})

print("pred_df created:", pred_df.shape)
pred_df.head()

pred_df created: (373033, 7)


,row_idx,y_true,y_logit,y_prob,cold_mask,scene_idx,alpha
0,2296852,0.0,0.321191,0.579614,False,2,1.0
1,495625,1.0,2.395261,0.916465,False,2,1.0
2,2437289,0.0,-2.105106,0.108602,False,3,1.0
3,1517928,0.0,-5.083009,0.006163,False,2,1.0
4,542839,1.0,0.955819,0.722284,False,2,1.0


Add derived columns (TP/FP/TN/FN, confidence, alpha bins, gates diagnostics)

In [17]:
# Derived columns
THRESH = 0.5  # keeping it fixed
y_pred = (pred_df["y_prob"].values >= THRESH).astype(np.int32)
y_true_int = pred_df["y_true"].values.astype(np.int32)

pred_df["y_pred"] = y_pred
pred_df["confidence"] = np.abs(pred_df["y_prob"].values - 0.5).astype(np.float32)

In [18]:
pred_df["is_tp"] = (y_pred == 1) & (y_true_int == 1)
pred_df["is_fp"] = (y_pred == 1) & (y_true_int == 0)
pred_df["is_tn"] = (y_pred == 0) & (y_true_int == 0)
pred_df["is_fn"] = (y_pred == 0) & (y_true_int == 1)

In [19]:
# Expert chosen (argmax) and max weight
pred_df["gate_expert"] = gates.argmax(axis=1).astype(np.int64)
pred_df["gate_max"] = gates.max(axis=1).astype(np.float32)

# Gate entropy (how "confident" the router is)
eps = 1e-12
p = np.clip(gates, eps, 1.0)
entropy = -(p * np.log(p)).sum(axis=1)  # natural log
pred_df["gate_entropy"] = entropy.astype(np.float32)

In [20]:
# Alpha bins (aligns with "meta-driven vs ID-driven" worked on notebook 09)
pred_df["alpha_bin"] = pd.cut(
    pred_df["alpha"],
    bins=[-np.inf, 0.33, 0.66, np.inf],
    labels=["alpha_low", "alpha_mid", "alpha_high"]
).astype(str)

In [21]:
print("Derived columns added.")
pred_df.head()

Derived columns added.


,row_idx,y_true,y_logit,y_prob,cold_mask,scene_idx,alpha,y_pred,confidence,is_tp,is_fp,is_tn,is_fn,gate_expert,gate_max,gate_entropy,alpha_bin
0,2296852,0.0,0.321191,0.579614,False,2,1.0,1,0.079614,False,True,False,False,0,0.316574,1.350762,alpha_high
1,495625,1.0,2.395261,0.916465,False,2,1.0,1,0.416465,True,False,False,False,0,0.316574,1.350762,alpha_high
2,2437289,0.0,-2.105106,0.108602,False,3,1.0,0,0.391398,False,False,True,False,2,0.650946,1.026001,alpha_high
3,1517928,0.0,-5.083009,0.006163,False,2,1.0,0,0.493837,False,False,True,False,0,0.316574,1.350762,alpha_high
4,542839,1.0,0.955819,0.722284,False,2,1.0,1,0.222284,True,False,False,False,0,0.316574,1.350762,alpha_high


#### Selection Table Summary

In [22]:
print("N total:", len(pred_df))
print("Cold rate:", float(pred_df["cold_mask"].mean()))
print("Positive rate:", float(pred_df["y_true"].mean()))

N total: 373033
Cold rate: 0.038975104079263766
Positive rate: 0.3367503583431244


In [23]:
print("--- Counts by cold/warm: ---\n")
print(pred_df.groupby("cold_mask")["y_true"].agg(["count", "mean"]).rename(columns={"mean": "pos_rate"}))

--- Counts by cold/warm: ---

            count  pos_rate
cold_mask                  
False      358494  0.344717
True        14539  0.140312


Cold-start interactions constitute a small but distinct subset (~3.9%) with a substantially lower positive rate, validating the need for specialised cold-start handling within the unified pipeline.

In [24]:
print("--- Counts by scenario: --- \n")
print(pred_df.groupby("scene_idx")["y_true"].agg(["count", "mean"]).rename(columns={"mean": "pos_rate"}))

--- Counts by scenario: --- 

            count  pos_rate
scene_idx                  
0           21682  0.041232
1           91798  0.231726
2          189948  0.467444
3           42514  0.297408
4           27091  0.074527


Interaction volume and outcome vary markedly across scenarios, justifying the use of scenario-aware routing rather than a single shared prediction pathway.

In [25]:
print("--- Counts by alpha_bin: ---\n")
print(pred_df.groupby("alpha_bin")["y_true"].agg(["count", "mean"]).rename(columns={"mean": "pos_rate"}))

--- Counts by alpha_bin: ---

             count  pos_rate
alpha_bin                   
alpha_high  365852  0.340821
alpha_low     4238  0.120340
alpha_mid     2943  0.142372


Most predictions rely heavily on ID-based embeddings (alpha_high), while lower-α cases correspond to colder items where meta-embeddings contribute more strongly.

In [26]:
print("--- Routing summary (expert selection counts): ---\n")
print(pred_df["gate_expert"].value_counts().sort_index())

--- Routing summary (expert selection counts): ---

gate_expert
0    189948
1     91798
2     91287
Name: count, dtype: int64


Multiple experts are actively selected at inference time, indicating that the MMoE router learns differentiated representations rather than collapsing to a single expert.

In [27]:
print("--- Gate max weight summary: ---\n")
print(pred_df["gate_max"].describe())

--- Gate max weight summary: ---

count    373033.000000
mean          0.369072
std           0.105901
min           0.284068
25%           0.316574
50%           0.316574
75%           0.386024
max           0.650946
Name: gate_max, dtype: float64


Gate confidence remains moderate (mean ≈ 0.37), suggesting soft expert mixtures rather than overconfident hard routing, which supports stable generalisation across scenarios.

In [28]:
print("--- Gate entropy summary: ---\n")
print(pred_df["gate_entropy"].describe())

--- Gate entropy summary: ---

count    373033.000000
mean          1.292807
std           0.104611
min           1.026001
25%           1.256635
50%           1.350762
75%           1.350762
max           1.379710
Name: gate_entropy, dtype: float64


High but non-maximal gate entropy indicates controlled uncertainty in expert selection, consistent with smooth scenario-aware routing rather than random or collapsed behaviour.

#### Define Explanation Cohorts (cold/warm + scenario + α regimes)

In [29]:
# Parameters
K_BATCH = 50     # SHAP batch per cohort
K_CASE  = 5      # case studies per cohort

# High-confidence threshold: use top quantile (to avoid arbitrary fixed cutoffs)
CONF_Q = 0.90  # top 10% most confident predictions within a cohort

BEST_SCENE = 1  # from Nb09: strong AUC
WORST_SCENE = 3 # from Nb09: weakest AUC

In [30]:
# Helper: sample with determinism
rng = np.random.default_rng(SEED)

In [31]:
def select_high_conf_subset(df, conf_q=0.90):
    """
    Returns df filtered to rows with confidence >= quantile(conf_q).
    Applied within each cohort to ensure "high-confidence" meaning is cohort-relative.
    """
    thr = df["confidence"].quantile(conf_q)
    return df[df["confidence"] >= thr], float(thr)

In [32]:
def pick_k_rows(df, k, sort_by="confidence", descending=True):
    """
    Deterministic selection:
    - if df larger than k: randomly sample (seeded) or take top-k (for case studies)
    """
    if len(df) == 0:
        return np.array([], dtype=np.int64)
    if sort_by is not None:
        df_sorted = df.sort_values(sort_by, ascending=not descending)
        return df_sorted["row_idx"].head(k).to_numpy(dtype=np.int64)
    # fallback random
    idx = df["row_idx"].to_numpy(dtype=np.int64)
    if len(idx) <= k:
        return idx
    return rng.choice(idx, size=k, replace=False)

In [33]:
def pick_k_random(df, k):
    """Seeded random selection for SHAP batches (avoids only extreme cases)."""
    if len(df) == 0:
        return np.array([], dtype=np.int64)
    idx = df["row_idx"].to_numpy(dtype=np.int64)
    if len(idx) <= k:
        return idx
    return rng.choice(idx, size=k, replace=False)

Build the cohorts and select K=50 + k=5 for each

In [34]:
cohorts = {}

def add_cohort(name, df_cohort):
    """
    For each cohort:
      - filter to high-confidence subset (top CONF_Q quantile)
      - pick K_BATCH random samples for SHAP batch
      - pick K_CASE top-confidence samples for case studies
    """
    df_hc, thr = select_high_conf_subset(df_cohort, conf_q=CONF_Q)

    batch_rows = pick_k_random(df_hc, K_BATCH)
    case_rows  = pick_k_rows(df_hc, K_CASE, sort_by="confidence", descending=True)

    cohorts[name] = {
        "cohort_size": int(len(df_cohort)),
        "high_conf_threshold": thr,
        "high_conf_size": int(len(df_hc)),
        "batch_row_idx": batch_rows.tolist(),
        "case_row_idx": case_rows.tolist(),
    }

Cold and Warm start

In [35]:
# Cold / High-confidence correct
cold_correct = pred_df[(pred_df["cold_mask"]) & ((pred_df["is_tp"]) | (pred_df["is_tn"]))]
add_cohort("cold_highconf_correct", cold_correct)

In [36]:
# Cold / High-confidence wrong
cold_wrong = pred_df[(pred_df["cold_mask"]) & ((pred_df["is_fp"]) | (pred_df["is_fn"]))]
add_cohort("cold_highconf_wrong", cold_wrong)

In [37]:
# Warm / High-confidence correct
warm_correct = pred_df[(~pred_df["cold_mask"]) & ((pred_df["is_tp"]) | (pred_df["is_tn"]))]
add_cohort("warm_highconf_correct", warm_correct)

In [38]:
# Warm / High-confidence wrong
warm_wrong = pred_df[(~pred_df["cold_mask"]) & ((pred_df["is_fp"]) | (pred_df["is_fn"]))]
add_cohort("warm_highconf_wrong", warm_wrong)

Scenario based

In [39]:
# Scenario-specific slices (best vs worst from Nb09)
#    Use all (not restricted to cold/warm), then high-confidence filter inside add_cohort.
scene_best = pred_df[pred_df["scene_idx"] == BEST_SCENE]
add_cohort(f"scene_{BEST_SCENE}_highconf", scene_best)

In [40]:
scene_worst = pred_df[pred_df["scene_idx"] == WORST_SCENE]
add_cohort(f"scene_{WORST_SCENE}_highconf", scene_worst)

Alpha values

In [41]:
# Alpha extremes
alpha_low = pred_df[pred_df["alpha_bin"] == "alpha_low"]
add_cohort("alpha_low_highconf", alpha_low)

In [42]:
alpha_high = pred_df[pred_df["alpha_bin"] == "alpha_high"]
add_cohort("alpha_high_highconf", alpha_high)

In [43]:
print("--- Explainability Cohorts Summary ---")
print(f"Selection: K_BATCH={K_BATCH} per cohort, K_CASE={K_CASE} per cohort")
print(f"High-confidence definition: top {int((1-CONF_Q)*100)}% by confidence within each cohort (quantile={CONF_Q})")
print(f"Scenario slices: BEST_SCENE={BEST_SCENE}, WORST_SCENE={WORST_SCENE}\n")

for name, info in cohorts.items():
    print(f"--- {name} ---")
    print(f"cohort_size:      {info['cohort_size']}")
    print(f"high_conf_thr:    {info['high_conf_threshold']:.4f}")
    print(f"high_conf_size:   {info['high_conf_size']}")
    print(f"batch_row_idx:    {len(info['batch_row_idx'])} selected")
    print(f"case_row_idx:     {len(info['case_row_idx'])} selected")
    print()

--- Explainability Cohorts Summary ---
Selection: K_BATCH=50 per cohort, K_CASE=5 per cohort
High-confidence definition: top 9% by confidence within each cohort (quantile=0.9)
Scenario slices: BEST_SCENE=1, WORST_SCENE=3

--- cold_highconf_correct ---
cohort_size:      12757
high_conf_thr:    0.4964
high_conf_size:   1276
batch_row_idx:    50 selected
case_row_idx:     5 selected

--- cold_highconf_wrong ---
cohort_size:      1782
high_conf_thr:    0.4536
high_conf_size:   179
batch_row_idx:    50 selected
case_row_idx:     5 selected

--- warm_highconf_correct ---
cohort_size:      304776
high_conf_thr:    0.4970
high_conf_size:   30478
batch_row_idx:    50 selected
case_row_idx:     5 selected

--- warm_highconf_wrong ---
cohort_size:      53718
high_conf_thr:    0.4209
high_conf_size:   5372
batch_row_idx:    50 selected
case_row_idx:     5 selected

--- scene_1_highconf ---
cohort_size:      91798
high_conf_thr:    0.4997
high_conf_size:   9180
batch_row_idx:    50 selected
case_ro

In [44]:
# Save JSON (reproducible selection indices for Notebook 10)
out_path = os.path.join(explain_dir, "explainability_sample_indices.json")
with open(out_path, "w") as f:
    json.dump({
        "seed": SEED,
        "K_BATCH": K_BATCH,
        "K_CASE": K_CASE,
        "CONF_Q": CONF_Q,
        "BEST_SCENE": BEST_SCENE,
        "WORST_SCENE": WORST_SCENE,
        "cohorts": cohorts
    }, f, indent=2)

Samples were stratified into cold/warm, correctness, scenario, and α-regime cohorts, and restricted to high-confidence predictions to ensure stable and interpretable SHAP attributions.

In [ ]:
!git config --global user.email "sharmfernando33@gmail.com"
!git config --global user.name "sheebaboy33"

Refresh index: 100% (14/14), done.
